In [1]:
#Convert cohort VCF to csv genotype table 

import vcfpy

In [2]:
reader = vcfpy.Reader.from_path("Pk_ten_bi.vcf.gz")

In [3]:
samples = reader.header.samples.names

In [4]:
types = {sample: {} for sample in samples}

In [ ]:
for record in reader:
    variant_id = f"{record.CHROM} {record.POS}"
    for call in record.calls:
        gt = call.data['GT']
        types[call.sample][variant_id] = gt

In [ ]:
import pandas as pd
df = pd.DataFrame.from_dict(types, orient='index')
df['Species'] = 'Pk'
df.head()

In [ ]:
df = df.reset_index().rename(columns={'index': 'Sample'})
df.to_csv('Pk_class_table.csv', index=False)

In [6]:
#subset tables to positions that are biallelic between species 

import vcfpy

reader = vcfpy.Reader.from_path("Pk_ten_bi.vcf.gz")

In [7]:
import pandas as pd

disc_snps = pd.read_csv('pkv_snps_fin.csv')

disc_snps['tag'] = disc_snps['tag'].str.replace('v2', 'v1')

In [8]:
disc_snps.head()

,Pk_pos,Pk_base,Pv_base,Pv_pos,buff,dist,frame_1,frame_2,contig,tag,Pk_seqid_short
0,31576,A,C,121212,4,31576,1,1,PKNH_01_v2,PvP01_01_v1,PKNH_01
1,31580,A,C,121216,2,31580,1,1,PKNH_01_v2,PvP01_01_v1,PKNH_01
2,31582,C,T,121218,2,31582,1,1,PKNH_01_v2,PvP01_01_v1,PKNH_01
3,31585,T,C,121221,3,31585,1,1,PKNH_01_v2,PvP01_01_v1,PKNH_01
4,31590,T,C,121226,2,31590,1,1,PKNH_01_v2,PvP01_01_v1,PKNH_01


In [9]:
pk_alt = {}

for record in reader:
    pk_alt[f"{record.CHROM} {record.POS}"] = record.ALT[0].value

In [10]:
pv_tab = pd.read_csv('pv_alt_pos.csv')

pk_tab = pd.read_csv('Fin_class_tab.csv')

In [11]:
pk_tab.head()

,Sample,PKNH_01_v2 53851,PKNH_01_v2 53971,PKNH_01_v2 53999,PKNH_01_v2 55164,PKNH_01_v2 55320,PKNH_01_v2 55790,PKNH_01_v2 58098,PKNH_01_v2 62058,PKNH_01_v2 62184,...,PKNH_API_v2 3774,PKNH_API_v2 17538,PKNH_API_v2 23330,PKNH_API_v2 24698,PKNH_API_v2 24749,PKNH_MIT_v2 2640,PKNH_MIT_v2 3785,PKNH_MIT_v2 3944,PKNH_MIT_v2 5329,Species
0,SAMEA104419839,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,Pk
1,SAMEA104419840,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,Pk
2,SAMEA104419841,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,Pk
3,SAMEA104419842,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,Pk
4,SAMEA104419843,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,Pk


In [12]:
pv_tab.shape

(150, 6973)

In [13]:
pk_red = pk_tab.drop(['Sample', 'Species'], axis=1)

In [15]:
Pv_ref = {}

for index, row in disc_snps.iterrows(): 
    Pk_pos = f"{row['contig']} {row['Pk_pos']}"
    Pv_bas = row['Pv_base']
    Pv_ref[Pk_pos]= Pv_bas

In [16]:
disc_snps.head()

,Pk_pos,Pk_base,Pv_base,Pv_pos,buff,dist,frame_1,frame_2,contig,tag,Pk_seqid_short
0,31576,A,C,121212,4,31576,1,1,PKNH_01_v2,PvP01_01_v1,PKNH_01
1,31580,A,C,121216,2,31580,1,1,PKNH_01_v2,PvP01_01_v1,PKNH_01
2,31582,C,T,121218,2,31582,1,1,PKNH_01_v2,PvP01_01_v1,PKNH_01
3,31585,T,C,121221,3,31585,1,1,PKNH_01_v2,PvP01_01_v1,PKNH_01
4,31590,T,C,121226,2,31590,1,1,PKNH_01_v2,PvP01_01_v1,PKNH_01


In [17]:
pk_red.shape

(279, 6971)

In [18]:
match_alt_cols = [] 

for column in pk_red.columns:
    pk_base_alt = pk_alt[column]
    pv_base = Pv_ref[column] 
    if pk_base_alt == pv_base: 
        match_alt_cols.append(column) 

In [19]:
len(match_alt_cols)

3840

In [20]:
sample = ['Sample']
match_alt_cols = sample + match_alt_cols
match_alt_cols.append('Species')

In [21]:
len(match_alt_cols)

3842

In [22]:
cross_bi = pk_tab[match_alt_cols]

In [43]:
reader = vcfpy.Reader.from_path("Pv_ten_snps.vcf.gz")

In [44]:
pv_alt = {}

for record in reader:
    pv_alt[f"{record.CHROM} {record.POS}"] = record.ALT[0].value

In [45]:
pv_red = pv_tab.drop(['Sample', 'Species'], axis=1)

In [46]:
Pk_ref = {}

for index, row in disc_snps.iterrows(): 
    Pv_pos = f"{row['tag']} {row['Pv_pos']}"
    Pk_bas = row['Pk_base']
    Pk_ref[Pv_pos]= Pk_bas

In [60]:
match_cols_pv = [] 

for column in pv_red.columns:
    pv_base_alt = pv_alt[column]
    pk_base = Pk_ref[column] 
    if pv_base_alt == pk_base: 
        match_cols_pv.append(column)

In [52]:
loci_snap = {}

for index, row in disc_snps.iterrows(): 
    Pv_pos = f"{row['tag']} {row['Pv_pos']}"
    Pk_pos = f"{row['contig']} {row['Pk_pos']}"
    loci_snap[Pv_pos]= Pk_pos

In [69]:
pk_equ_snap = []

for item in match_cols_pv: 
    pk_equ = loci_snap[item]
    pk_equ_snap.append(pk_equ)

In [70]:
sample = ['Sample']
pk_equ_snap = sample + pk_equ_snap
pk_equ_snap.append('Species')

final_shared_cols = []

for item in cross_bi.columns: 
    if item in pk_equ_snap: 
        final_shared_cols.append(item)

In [73]:
cross_bi_fin = cross_bi[final_shared_cols]

In [74]:
cross_bi_fin = cross_bi_fin.set_index('Sample')

In [79]:
cross_bi_fin.to_csv('Ult_class_tab.csv')